# 🐍 Notebook 08: Object-Oriented Programming, Exceptions & File I/O
**Trainer: Sundeep Hiremat · TASK Nalgonda Center**  
**Course:** Cloud Software Developer (AWS DVA-C02 Aligned)

In this notebook, we complete the core Python foundation:
1. **Object-Oriented Programming (OOP):** Modeling cloud resources with Classes, Inheritance, and Encapsulation.
2. **Exception Handling:** Building bulletproof code with `try / except / else / finally`.
3. **File Handling & JSON:** Managing local configuration files and AWS event payloads.


## 1. Object-Oriented Programming (OOP) for Cloud
We model cloud assets (EC2, S3, RDS) as objects with states and behaviors.


In [ ]:
# 1. Base Class (Parent)
class CloudResource:
    """Base class representing any AWS cloud resource."""
    cloud_provider = "AWS"  # Class attribute
    
    def __init__(self, resource_id: str, region: str = "ap-south-1"):
        self.resource_id = resource_id  # Public attribute
        self.region = region
        self._state = "PENDING"         # Protected attribute
        self.__secret_arn = f"arn:aws:secret:{resource_id}" # Private attribute
        
    def start(self):
        self._state = "RUNNING"
        print(f"🟢 Resource {self.resource_id} is now RUNNING in {self.region}.")
        
    def stop(self):
        self._state = "STOPPED"
        print(f"🔴 Resource {self.resource_id} is now STOPPED.")
        
    def get_status(self):
        return self._state

# 2. Subclass / Inheritance (Child)
class EC2Instance(CloudResource):
    """Inherits from CloudResource and adds compute-specific features."""
    def __init__(self, resource_id: str, instance_type: str = "t3.micro", region: str = "ap-south-1"):
        super().__init__(resource_id, region)
        self.instance_type = instance_type
        
    # Method Overriding (Polymorphism)
    def start(self):
        super().start()
        print(f"  ⚡ Booting virtual CPU on {self.instance_type} compute core...")

# Create and test
server = EC2Instance("i-04a29c7811", instance_type="t3.medium")
server.start()
print("State:", server.get_status())
server.stop()


---
## 2. Robust Exception Handling (`try / except / else / finally`)
Cloud operations must handle missing resources, API timeouts, and quota limits gracefully.


In [ ]:
# Custom Domain Exception
class BudgetExceededError(Exception):
    """Raised when cloud spending exceeds authorized monthly cap."""
    pass

def provision_fleet(server_count: int, unit_cost: float, budget_limit: float):
    try:
        if server_count <= 0:
            raise ValueError("Server count must be at least 1!")
            
        total_cost = server_count * unit_cost
        if total_cost > budget_limit:
            raise BudgetExceededError(
                f"Cannot launch {server_count} instances! Total: ${total_cost}, Cap: ${budget_limit}"
            )
            
        print(f"🎉 Successfully provisioned {server_count} instances for ${total_cost}.")
        return True

    except ValueError as ve:
        print(f"❌ Input Validation Error: {ve}")
        return False
    except BudgetExceededError as be:
        print(f"🛑 Cost Shield Triggered: {be}")
        return False
    except Exception as e:
        print(f"💥 Unexpected Error: {e}")
        return False
    else:
        print("✅ Execution completed without any errors.")
    finally:
        print("🔒 [Audit System] Provisioning attempt logged to CloudTrail.")

# Test scenarios
provision_fleet(server_count=10, unit_cost=50.0, budget_limit=300.0) # Triggers budget error
print("-" * 50)
provision_fleet(server_count=3, unit_cost=50.0, budget_limit=300.0)  # Succeeds


---
## 3. File I/O & JSON Serialization (Cloud Configs)
In AWS, configuration files, IAM policies, and Lambda event triggers are all JSON.


In [ ]:
import json

# Python Dictionary representing cloud architecture
deployment_manifest = {
    "Project": "Cloud Portfolio",
    "Environment": "Production",
    "Center": "TASK Nalgonda",
    "Trainer": "Sundeep Hiremat",
    "EC2_Instances": [
        {"id": "i-web-01", "type": "t3.micro", "public_ip": "13.233.45.10"},
        {"id": "i-web-02", "type": "t3.micro", "public_ip": "13.233.45.11"}
    ],
    "SecurityGroup": {
        "Name": "web-sg",
        "InboundPorts": [80, 443, 22]
    }
}

# 1. Writing to JSON file using context manager ('with')
filename = "cloud_manifest.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(deployment_manifest, f, indent=4)
print(f"✅ Created '{filename}' on disk.")

# 2. Reading back and deserializing
with open(filename, "r", encoding="utf-8") as f:
    loaded_config = json.load(f)

print("\n📂 Loaded from JSON:")
print(f"Project: {loaded_config['Project']} by {loaded_config['Trainer']}")
print(f"Open Inbound Ports: {loaded_config['SecurityGroup']['InboundPorts']}")
